# MAPPO Training — Fix 5

Multi-Agent PPO for UAV crop-disease monitoring (`uav_env_5` / `networks_5`).

**Fixes over notebook Fix 4:**
- Disease dynamics strengthened (more seeds, higher spread, re-seeding)
- Per-UAV critic heads — each actor uses its own baseline, not pooled mean-reward
- Reward normalisation: per-episode std scaling only (no global mean subtraction)
- Reward clip ±50 to suppress outlier return spikes
- KL early-stopping (0.02) inside PPO inner loop
- K_EPOCHS = 2, entropy annealed 0.01 → 0.001
- Post-crash steps masked from per-UAV advantage
- Strict hover threshold (HOVER_MAG=0.2), overhover grace step, IMMUNITY_DAYS timer

In [ ]:
import os

print('Files in /kaggle/input:')
print('-' * 40)
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


## Setup

In [ ]:
import os, sys

ON_KAGGLE = os.path.exists('/kaggle/input')

if ON_KAGGLE:
    # Upload uav_env_5.py and networks_5.py as a Kaggle dataset
    # and reference it below (adjust the dataset slug to match)
    MODULE_DIR  = '/kaggle/input/uav-fix5'
    WEIGHTS_DIR = '/kaggle/working/weights_fix5'
    RESULTS_DIR = '/kaggle/working/results_fix5'
else:
    MODULE_DIR  = os.path.dirname(os.path.abspath('__file__'))
    ROOT_DIR    = os.path.dirname(os.path.dirname(MODULE_DIR))
    WEIGHTS_DIR = os.path.join(ROOT_DIR, 'weights_fix5')
    RESULTS_DIR = os.path.join(ROOT_DIR, 'results_fix5')

if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR)

CKPT_DIR = os.path.join(WEIGHTS_DIR, 'checkpoints')
BEST_DIR = os.path.join(WEIGHTS_DIR, 'best')
for d in (WEIGHTS_DIR, RESULTS_DIR, CKPT_DIR, BEST_DIR):
    os.makedirs(d, exist_ok=True)

print('MODULE_DIR  :', MODULE_DIR)
print('WEIGHTS_DIR :', WEIGHTS_DIR)
print('RESULTS_DIR :', RESULTS_DIR)


In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from collections import deque
from tqdm.auto import tqdm

from uav_env_5 import (
    UAVFieldEnv, N_UAVS, N_SECTORS, OBS_SIZE, DAILY_STEPS_MAX,
    GRID_ROWS, GRID_COLS,
)
from networks_5 import SectorAttentionActor, CriticNetwork, JOINT_SIZE

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device     : {DEVICE}')
print(f'OBS_SIZE   : {OBS_SIZE}')
print(f'JOINT_SIZE : {JOINT_SIZE}')


## Hyperparameters

In [ ]:
N_EPISODES      = 400
K_EPOCHS        = 2         # reduced from 4 to limit per-update overshoot
MINI_BATCH_SIZE = 128
GAMMA           = 0.99      # RL discount (not env GAMMA=0.8 history decay)
GAE_LAMBDA      = 0.95
EPS_CLIP        = 0.2
ENTROPY_START   = 0.01      # annealed to ENTROPY_END over training
ENTROPY_END     = 0.001
LR_ACTOR        = 3e-4
LR_CRITIC       = 3e-4
LR_MIN          = 1e-5
KL_EARLY_STOP   = 0.02      # break PPO inner loop if approx KL exceeds this
REWARD_CLIP     = 50.0      # clip individual rewards before GAE
SAVE_INTERVAL   = 25
LOG_INTERVAL    = 5


## Environment & Model Initialisation

In [ ]:
env = UAVFieldEnv()

actors = [SectorAttentionActor().to(DEVICE) for _ in range(N_UAVS)]
critic = CriticNetwork().to(DEVICE)

actor_opts = [torch.optim.Adam(a.parameters(), lr=LR_ACTOR) for a in actors]
critic_opt  = torch.optim.Adam(critic.parameters(), lr=LR_CRITIC)

actor_scheds = [
    torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=N_EPISODES, eta_min=LR_MIN)
    for opt in actor_opts
]
critic_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
    critic_opt, T_max=N_EPISODES, eta_min=LR_MIN
)

def count_params(m):
    return sum(p.numel() for p in m.parameters())

print(f'Actor params/UAV : {count_params(actors[0]):>10,}  x{N_UAVS} = {count_params(actors[0])*N_UAVS:,}')
print(f'Critic params    : {count_params(critic):>10,}')
print(f'Total params     : {count_params(actors[0])*N_UAVS + count_params(critic):,}')


## Training Components

In [ ]:
class RolloutBuffer:
    def __init__(self):
        self.clear()

    def clear(self):
        self.obs        = [[] for _ in range(N_UAVS)]
        self.actions    = [[] for _ in range(N_UAVS)]
        self.log_probs  = [[] for _ in range(N_UAVS)]
        self.rewards    = [[] for _ in range(N_UAVS)]
        self.alive_mask = [[] for _ in range(N_UAVS)]
        self.values     = []   # (T, N_UAVS) per-UAV values
        self.dones      = []

    def store(self, obs_list, actions_list, log_probs_list, rewards_list,
              alive_list, values_row, done):
        for u in range(N_UAVS):
            self.obs[u].append(obs_list[u])
            self.actions[u].append(actions_list[u])
            self.log_probs[u].append(log_probs_list[u])
            self.rewards[u].append(rewards_list[u])
            self.alive_mask[u].append(alive_list[u])
        self.values.append(values_row)
        self.dones.append(done)

    def to_tensors(self):
        obs_t   = [torch.as_tensor(np.asarray(self.obs[u]),     dtype=torch.float32, device=DEVICE)
                   for u in range(N_UAVS)]
        acts_t  = [torch.as_tensor(np.asarray(self.actions[u]), dtype=torch.float32, device=DEVICE)
                   for u in range(N_UAVS)]
        lps_t   = [torch.stack(self.log_probs[u]).squeeze(-1).to(DEVICE)
                   for u in range(N_UAVS)]
        rews_t  = [torch.as_tensor(self.rewards[u], dtype=torch.float32, device=DEVICE)
                   for u in range(N_UAVS)]
        alive_t = [torch.as_tensor(self.alive_mask[u], dtype=torch.float32, device=DEVICE)
                   for u in range(N_UAVS)]
        vals_t  = torch.stack([v.view(N_UAVS) for v in self.values]).to(DEVICE)  # (T, N_UAVS)
        dones_t = torch.as_tensor(self.dones, dtype=torch.float32, device=DEVICE)
        return obs_t, acts_t, lps_t, rews_t, alive_t, vals_t, dones_t


In [ ]:
def compute_gae(rewards_per_uav, values_per_uav, dones, last_values_per_uav, alive_per_uav):
    """
    Per-UAV GAE using per-UAV critic heads.
    Post-crash steps survive in returns but are masked during advantage
    normalisation and PPO loss so they contribute zero gradient.
    """
    t_ep        = values_per_uav.shape[0]
    per_uav_adv = []
    per_uav_ret = []

    for u in range(N_UAVS):
        values_ext = torch.cat([values_per_uav[:, u], last_values_per_uav[u].view(1)])
        adv_u = torch.zeros(t_ep, device=DEVICE)
        gae   = 0.0
        for t in reversed(range(t_ep)):
            mask  = 1.0 - dones[t]
            delta = rewards_per_uav[u][t] + GAMMA * values_ext[t + 1] * mask - values_ext[t]
            gae   = delta + GAMMA * GAE_LAMBDA * mask * gae
            adv_u[t] = gae
        returns_u = adv_u + values_per_uav[:, u]

        # Normalise only over alive (non-crashed) timesteps
        alive = alive_per_uav[u] > 0.5
        if alive.any():
            adv_u = (adv_u - adv_u[alive].mean()) / (adv_u[alive].std() + 1e-8)
        per_uav_adv.append(adv_u.detach())
        per_uav_ret.append(returns_u.detach())

    returns = torch.stack(per_uav_ret, dim=1)   # (T, N_UAVS)
    return per_uav_adv, returns


In [ ]:
def ppo_update(env, actors, critic, actor_opts, critic_opt, buffer, entropy_coeff):
    obs_t, acts_t, old_lps_t, rews_t, alive_t, vals_t, dones_t = buffer.to_tensors()

    with torch.no_grad():
        last_joint = torch.cat(
            [torch.as_tensor(env._get_obs(u), dtype=torch.float32, device=DEVICE)
             for u in range(N_UAVS)]
        ).unsqueeze(0)
        last_values = critic(last_joint).view(N_UAVS)

    per_uav_adv, returns = compute_gae(
        rews_t, vals_t, dones_t, last_values, alive_t
    )

    t_ep          = vals_t.shape[0]
    actor_losses  = []
    critic_losses = []
    approx_kls    = []
    early_stopped = False

    for _ in range(K_EPOCHS):
        if early_stopped:
            break
        perm = torch.randperm(t_ep, device=DEVICE)
        for start in range(0, t_ep, MINI_BATCH_SIZE):
            mb_idx = perm[start : start + MINI_BATCH_SIZE]

            # ── Actor updates (one per UAV) ────────────────────────────
            kl_batch, n_kl = 0.0, 0
            for u in range(N_UAVS):
                mb_alive = alive_t[u][mb_idx]
                if mb_alive.sum() < 1:
                    continue
                mb_obs     = obs_t[u][mb_idx]
                mb_actions = acts_t[u][mb_idx]
                mb_old_lps = old_lps_t[u][mb_idx].detach()
                mb_adv     = per_uav_adv[u][mb_idx]

                new_lps, entropy = actors[u].get_log_prob_entropy(mb_obs, mb_actions)
                ratio  = torch.exp(new_lps - mb_old_lps)
                surr1  = ratio * mb_adv
                surr2  = torch.clamp(ratio, 1 - EPS_CLIP, 1 + EPS_CLIP) * mb_adv
                a_loss = (-torch.min(surr1, surr2) * mb_alive).sum() / mb_alive.sum()
                a_loss = a_loss - entropy_coeff * entropy

                actor_opts[u].zero_grad()
                a_loss.backward()
                nn.utils.clip_grad_norm_(actors[u].parameters(), 0.5)
                actor_opts[u].step()
                actor_losses.append(a_loss.item())

                with torch.no_grad():
                    kl_u = ((mb_old_lps - new_lps) * mb_alive).sum() / mb_alive.sum().clamp(min=1.0)
                kl_batch += kl_u.item(); n_kl += 1

            if n_kl > 0:
                avg_kl = kl_batch / n_kl
                approx_kls.append(avg_kl)
                if avg_kl > KL_EARLY_STOP:
                    early_stopped = True

            # ── Critic update (per-head MSE) ───────────────────────────
            mb_joint    = torch.cat([obs_t[u][mb_idx] for u in range(N_UAVS)], dim=1)
            values_pred = critic(mb_joint)          # (B, N_UAVS)
            mb_returns  = returns[mb_idx]            # (B, N_UAVS)
            c_loss      = nn.MSELoss()(values_pred, mb_returns)

            critic_opt.zero_grad()
            c_loss.backward()
            nn.utils.clip_grad_norm_(critic.parameters(), 0.5)
            critic_opt.step()
            critic_losses.append(c_loss.item())

            if early_stopped:
                break

    kl_mean = float(np.mean(approx_kls)) if approx_kls else 0.0
    return (float(np.mean(actor_losses)) if actor_losses else 0.0,
            float(np.mean(critic_losses)) if critic_losses else 0.0,
            kl_mean, early_stopped)


## Training Loop

In [ ]:
def entropy_coeff(ep):
    frac = ep / max(1, N_EPISODES - 1)
    return ENTROPY_START + (ENTROPY_END - ENTROPY_START) * frac

buffer          = RolloutBuffer()
ep_rewards      = []
ep_diagnosed    = []
ep_detect_rates = []
ep_crashes      = []
ep_disease_max  = []
a_loss_log      = []
c_loss_log      = []
kl_log          = []
recent_rews     = deque(maxlen=50)
best_diag       = 0
train_start     = time.time()

pbar = tqdm(range(1, N_EPISODES + 1), desc='Training', unit='ep')
for episode in pbar:
    obs            = env.reset()
    buffer.clear()
    ep_reward      = 0.0
    ep_crash_count = 0
    max_true_inf   = 0

    # ── Collect episode ───────────────────────────────────────────────
    for _ in range(env.total_steps):
        joint_obs = torch.cat(
            [torch.as_tensor(obs[u], dtype=torch.float32, device=DEVICE)
             for u in range(N_UAVS)]
        ).unsqueeze(0)

        with torch.no_grad():
            values_row = critic(joint_obs).squeeze(0)   # (N_UAVS,)

        alive    = [0.0 if env.crashed[u] else 1.0 for u in range(N_UAVS)]
        actions  = []
        log_probs = []
        for u in range(N_UAVS):
            obs_t = torch.as_tensor(obs[u], dtype=torch.float32, device=DEVICE)
            with torch.no_grad():
                action, lp = actors[u].get_action(obs_t)
            actions.append(action)
            log_probs.append(lp)

        next_obs, rewards, done, info = env.step(actions)
        ep_crash_count += sum(info['newly_crashed'])
        max_true_inf    = max(max_true_inf, int(info['true_status'].sum()))

        # Clip individual rewards before storing
        rewards = [float(np.clip(r, -REWARD_CLIP, REWARD_CLIP)) for r in rewards]

        buffer.store(obs, actions, log_probs, rewards, alive, values_row, float(done))
        ep_reward += float(sum(rewards))
        obs = next_obs
        if done:
            break

    # ── Per-episode reward scaling (std only, no mean subtract) ──────
    all_rews = np.concatenate(
        [np.asarray(buffer.rewards[u], dtype=np.float32) for u in range(N_UAVS)]
    )
    scale = max(float(all_rews.std()), 1.0)
    for u in range(N_UAVS):
        buffer.rewards[u] = [r / scale for r in buffer.rewards[u]]

    # ── PPO update ────────────────────────────────────────────────────
    ent_c = entropy_coeff(episode)
    a_loss, c_loss, kl, stopped = ppo_update(
        env, actors, critic, actor_opts, critic_opt, buffer, ent_c
    )

    for sched in actor_scheds:
        sched.step()
    critic_sched.step()

    # ── Metrics ───────────────────────────────────────────────────────
    n_diag     = int((env.uav_status != 2).sum())
    n_ever_inf = int(env.ever_infected.sum())
    n_detected = int((env.ever_infected & env.ever_diagnosed).sum())
    detect_rate = n_detected / max(n_ever_inf, 1)

    ep_rewards.append(ep_reward)
    ep_diagnosed.append(n_diag)
    ep_detect_rates.append(detect_rate)
    ep_crashes.append(ep_crash_count)
    ep_disease_max.append(max_true_inf)
    a_loss_log.append(a_loss)
    c_loss_log.append(c_loss)
    kl_log.append(kl)
    recent_rews.append(ep_reward)

    # ── Best checkpoint (by sectors diagnosed) ────────────────────────
    if n_diag > best_diag:
        best_diag = n_diag
        for u in range(N_UAVS):
            torch.save(actors[u].state_dict(),
                       os.path.join(BEST_DIR, f'actor{u}_best.pth'))
        torch.save(critic.state_dict(), os.path.join(BEST_DIR, 'critic_best.pth'))

    # ── Periodic checkpoint ───────────────────────────────────────────
    if episode % SAVE_INTERVAL == 0:
        for u in range(N_UAVS):
            torch.save(actors[u].state_dict(),
                       os.path.join(CKPT_DIR, f'actor{u}_ep{episode}.pth'))
        torch.save(critic.state_dict(),
                   os.path.join(CKPT_DIR, f'critic_ep{episode}.pth'))

    # ── Progress bar ──────────────────────────────────────────────────
    avg50 = float(np.mean(recent_rews))
    pbar.set_postfix({
        'rew':   f'{ep_reward:.1f}',
        'avg50': f'{avg50:.1f}',
        'diag':  f'{n_diag}/{N_SECTORS}',
        'det%':  f'{detect_rate*100:.0f}%',
        'maxI':  max_true_inf,
        'crash': ep_crash_count,
        'kl':    f'{kl:.3f}',
        'es':    'Y' if stopped else '.'
    })

    if episode % LOG_INTERVAL == 0 or episode == 1:
        elapsed = time.time() - train_start
        eta_min = elapsed / episode * (N_EPISODES - episode) / 60
        tqdm.write(
            f'Ep {episode:>4}/{N_EPISODES}  '
            f'rew={ep_reward:>9.2f}  avg50={avg50:>9.2f}  '
            f'diag={n_diag:>3}/{N_SECTORS}  '
            f'det={detect_rate*100:5.1f}%  ({n_detected}/{n_ever_inf})  '
            f'maxI={max_true_inf:>3}  crash={ep_crash_count:>3}  '
            f'a={a_loss:.4f}  c={c_loss:.4f}  kl={kl:.3f}  '
            f'ent={ent_c:.4f}  lr={actor_opts[0].param_groups[0]["lr"]:.2e}  '
            f'ETA={eta_min:.1f}min'
        )

total_time = time.time() - train_start
print(f'\nTraining complete in {total_time/60:.1f} min')
print(f'Best diagnosed  : {best_diag} / {N_SECTORS}')
print(f'Best detect rate: {max(ep_detect_rates)*100:.1f}%')
print(f'Total crashes   : {sum(ep_crashes)}')


## Training Results

In [ ]:
def moving_avg(data, w=50):
    return np.convolve(data, np.ones(w) / w, mode='valid')

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
fig.suptitle('MAPPO Training — Fix 5', fontsize=13)

# Row 0
axes[0, 0].plot(ep_rewards, alpha=0.2, color='steelblue')
if len(ep_rewards) >= 50:
    axes[0, 0].plot(moving_avg(ep_rewards), color='steelblue', label='MA-50'); axes[0, 0].legend()
axes[0, 0].set_title('Episode Reward'); axes[0, 0].set_xlabel('Episode'); axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(ep_diagnosed, alpha=0.2, color='tomato')
if len(ep_diagnosed) >= 50:
    axes[0, 1].plot(moving_avg(ep_diagnosed), color='tomato', label='MA-50'); axes[0, 1].legend()
axes[0, 1].axhline(N_SECTORS, color='k', ls='--', lw=0.8)
axes[0, 1].set_title('Sectors Diagnosed / Episode'); axes[0, 1].set_xlabel('Episode'); axes[0, 1].grid(True, alpha=0.3)

axes[0, 2].plot(ep_disease_max, alpha=0.2, color='darkred')
if len(ep_disease_max) >= 50:
    axes[0, 2].plot(moving_avg(ep_disease_max), color='darkred', label='MA-50'); axes[0, 2].legend()
axes[0, 2].set_title('Peak True Infected / Episode'); axes[0, 2].set_xlabel('Episode'); axes[0, 2].grid(True, alpha=0.3)

# Row 1
pct = [r * 100 for r in ep_detect_rates]
axes[1, 0].plot(pct, alpha=0.2, color='darkorange')
if len(pct) >= 50:
    axes[1, 0].plot(moving_avg(pct), color='darkorange', label='MA-50'); axes[1, 0].legend()
axes[1, 0].axhline(100, color='k', ls='--', lw=0.8); axes[1, 0].set_ylim(0, 105)
axes[1, 0].set_title('Disease Detection Rate (%)'); axes[1, 0].set_xlabel('Episode')
axes[1, 0].set_ylabel('%'); axes[1, 0].grid(True, alpha=0.3)

if len(a_loss_log) >= 50:
    axes[1, 1].plot(moving_avg(a_loss_log), color='firebrick')
axes[1, 1].set_title('Actor Loss (MA-50)'); axes[1, 1].set_xlabel('Episode'); axes[1, 1].grid(True, alpha=0.3)

if len(c_loss_log) >= 50:
    axes[1, 2].plot(moving_avg(c_loss_log), color='mediumseagreen')
axes[1, 2].set_title('Critic Loss (MA-50)'); axes[1, 2].set_xlabel('Episode'); axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
out_path = os.path.join(RESULTS_DIR, 'training_curves_fix5.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print('Saved:', out_path)


## Save Final Weights

In [ ]:
for u in range(N_UAVS):
    path = os.path.join(WEIGHTS_DIR, f'actor{u}_final.pth')
    torch.save(actors[u].state_dict(), path)
    print(f'Saved actor {u} -> {path}')

path = os.path.join(WEIGHTS_DIR, 'critic_final.pth')
torch.save(critic.state_dict(), path)
print(f'Saved critic  -> {path}')
print(f'Best weights  -> {BEST_DIR}/')


## Evaluation

Loads the best-checkpoint weights and runs a single deterministic episode (mean action from the actor).

In [ ]:
eval_actors = [SectorAttentionActor().to(DEVICE) for _ in range(N_UAVS)]
for u in range(N_UAVS):
    path = os.path.join(BEST_DIR, f'actor{u}_best.pth')
    eval_actors[u].load_state_dict(torch.load(path, map_location=DEVICE))
    eval_actors[u].eval()
    std = torch.exp(torch.clamp(eval_actors[u].action_log_std, -2.0, 0.5)).detach().cpu().numpy()
    print(f'Loaded actor {u}  action_std={std}')


In [ ]:
eval_env  = UAVFieldEnv(seed=0)
obs       = eval_env.reset()
eval_hist = []
ep_rew    = 0.0

for _ in range(eval_env.total_steps):
    actions = []
    for u in range(N_UAVS):
        obs_t = torch.as_tensor(obs[u], dtype=torch.float32, device=DEVICE).unsqueeze(0)
        with torch.no_grad():
            action = eval_actors[u](obs_t).loc.squeeze(0).cpu().numpy()
        actions.append(action)
    obs, rewards, done, info = eval_env.step(actions)
    ep_rew += sum(rewards)
    eval_hist.append(info)
    if done:
        break

print(f'Eval episode reward : {ep_rew:.2f}')


In [ ]:
timesteps   = [s['t'] for s in eval_hist]
n_visited   = [(s['uav_status'] != 2).sum() for s in eval_hist]
n_inf_known = [(s['uav_status'] == 1).sum() for s in eval_hist]
n_true_inf  = [s['true_status'].sum()       for s in eval_hist]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Evaluation Episode (deterministic policy)', fontsize=12)

axes[0].plot(timesteps, n_visited, color='steelblue')
axes[0].set_title('Sectors Diagnosed'); axes[0].set_xlabel('Step')
axes[0].axhline(N_SECTORS, color='k', ls='--', lw=0.8); axes[0].grid(True, alpha=0.3)

axes[1].plot(timesteps, n_true_inf,  color='tomato',   label='True infected')
axes[1].plot(timesteps, n_inf_known, color='orange', ls='--', label='UAV-known infected')
axes[1].set_title('Infection Tracking'); axes[1].set_xlabel('Step')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(timesteps, [s['energy'][0] for s in eval_hist], label='UAV 0')
axes[2].plot(timesteps, [s['energy'][1] for s in eval_hist], label='UAV 1')
axes[2].plot(timesteps, [s['energy'][2] for s in eval_hist], label='UAV 2')
axes[2].plot(timesteps, [s['energy'][3] for s in eval_hist], label='UAV 3')
axes[2].set_title('UAV Energy'); axes[2].set_xlabel('Step')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
out_path = os.path.join(RESULTS_DIR, 'eval_fix5.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print('Saved:', out_path)


In [ ]:
final   = eval_hist[-1]
n_diag  = int((final['uav_status'] != 2).sum())
n_inf_k = int((final['uav_status'] == 1).sum())
n_true  = int(final['true_status'].sum())
n_crash = sum(1 for h in eval_hist if any(h['newly_crashed']))
n_ever_inf  = int(eval_env.ever_infected.sum())
n_detected  = int((eval_env.ever_infected & eval_env.ever_diagnosed).sum())

print('=' * 55)
print('  EVALUATION SUMMARY  (Fix 5)')
print('=' * 55)
print(f'  Grid              : {GRID_ROWS}x{GRID_COLS} = {N_SECTORS} sectors')
print(f'  Episode reward    : {ep_rew:.2f}')
print(f'  Sectors diagnosed : {n_diag} / {N_SECTORS}  ({100*n_diag/N_SECTORS:.1f}%)')
print(f'  Detection rate    : {n_detected}/{n_ever_inf}  ({100*n_detected/max(n_ever_inf,1):.1f}%)')
print(f'  UAV-known infected: {n_inf_k}')
print(f'  True infected     : {n_true}')
print(f'  Crash steps       : {n_crash}')
print(f'  Best train diag   : {best_diag} / {N_SECTORS}')
print('=' * 55)
